# Graph RAG — Implementation Template

## What is Graph RAG?
Standard RAG treats every chunk as isolated. Graph RAG adds **edges** between
related chunks so that retrieving one node also pulls in its neighbours.

For FastAPI code this matters because:
- `read_item()` and `create_item()` often appear in the same tutorial file
- `OAuth2PasswordBearer.__init__` and `__call__` are always used together
- `Depends()` in a route only makes sense alongside the dependency function

## How it fits into the existing system
This notebook adds one new class: `GraphRAG`.  
It **wraps** `VersionControlRAG` — everything already working stays unchanged.

```
retrieve_graph(query, top_k, mode)  →  list[str]
```
Same return type as `retrieve_complex()` — drop-in replacement for eval.

## Your tasks
| Step | Status | What to do |
|---|---|---|
| 0 · Install | ready | Run once |
| 1 · Load | ready | Run as-is |
| 2 · Build graph | **partial** | Uncomment `same_class` + `calls` skeletons |
| 3 · GraphRAG class | ready | Run as-is |
| 4 · Quick test | ready | Plug in your Pinecone key and run |
| 5 · Eval | ready | Adds to `chunking_test.ipynb` automatically |

## 0 · Install

In [ ]:
%pip install -q networkx sentence-transformers rank_bm25

## 1 · Imports

In [ ]:
import re
import json
import networkx as nx
from pathlib import Path
from collections import defaultdict

import rag_pipeline, importlib
importlib.reload(rag_pipeline)
from rag_pipeline import VersionControlRAG

DATA_DIR = Path(".")

with open(DATA_DIR / "local_corpus.json") as f:
    corpus = json.load(f)

print(f"Corpus loaded: {len(corpus)} chunks")

## 2 · Build the code graph

Each **node** = one corpus chunk (identified by its `doc_id`)  
Each **edge** = a relationship between two chunks

| Edge type | Meaning | Weight |
|---|---|---|
| `same_file` | Both chunks come from the same `.py` file | 0.9 |
| `same_topic` | Both from the same tutorial topic folder | 0.6 |
| `same_class` | Both are methods of the same Python class | 0.8 |
| `calls` | Function A calls function B by name | 0.7 |

**`same_file` and `same_topic` are already implemented.**  
**Your job: uncomment the `same_class` and `calls` skeletons below.**

In [ ]:
def build_code_graph(corpus: list[dict]) -> nx.Graph:
    G = nx.Graph()

    # ── Add all nodes ─────────────────────────────────────────────────────────
    for item in corpus:
        G.add_node(
            item["id"],
            content=item["content"],
            tier=item["metadata"].get("tier", "docs_src")
        )

    # ── Helpers ───────────────────────────────────────────────────────────────
    def source_file(doc_id):
        return re.sub(r"_f\d+$", "", doc_id)

    def topic(doc_id):
        m = re.search(r"docs_src_([^_]+)", doc_id)
        return m.group(1) if m else ""

    def class_context(content):
        """Return the first class name found in the content, or None."""
        m = re.search(r"class\s+(\w+)", content)
        return m.group(1) if m else None

    # ── EDGE TYPE 1: same_file (implemented) ──────────────────────────────────
    by_file = defaultdict(list)
    for item in corpus:
        by_file[source_file(item["id"])].append(item["id"])

    for file_chunks in by_file.values():
        for i, a in enumerate(file_chunks):
            for b in file_chunks[i + 1:]:
                G.add_edge(a, b, type="same_file", weight=0.9)

    # ── EDGE TYPE 2: same_topic (implemented) ─────────────────────────────────
    by_topic = defaultdict(list)
    for item in corpus:
        t = topic(item["id"])
        if t:
            by_topic[t].append(item["id"])

    for topic_chunks in by_topic.values():
        for i, a in enumerate(topic_chunks):
            for b in topic_chunks[i + 1:]:
                if not G.has_edge(a, b):
                    G.add_edge(a, b, type="same_topic", weight=0.6)

    # ── EDGE TYPE 3: same_class  ◄─── YOUR TASK ──────────────────────────────
    # Uncomment and complete the skeleton below.
    # Goal: connect methods that belong to the same Python class.
    # Hint: scan each chunk for 'def method(self,' to detect methods,
    #       then use class_context() to find the class name.
    #
    # by_class = defaultdict(list)
    # for item in corpus:
    #     cls = class_context(item["content"])
    #     if cls:
    #         by_class[cls].append(item["id"])
    # for cls_chunks in by_class.values():
    #     for i, a in enumerate(cls_chunks):
    #         for b in cls_chunks[i + 1:]:
    #             if not G.has_edge(a, b):
    #                 G.add_edge(a, b, type="same_class", weight=0.8)

    # ── EDGE TYPE 4: calls  ◄─── YOUR TASK ───────────────────────────────────
    # Uncomment and complete the skeleton below.
    # Goal: connect chunk A to chunk B when A's code calls B's function.
    # Hint: build a {fn_name: doc_id} map, then scan each chunk's content
    #       for function-call patterns like 'fn_name('.
    #
    # fn_name_to_id = {}
    # for item in corpus:
    #     m = re.search(r"(?:async )?def (\w+)\(", item["content"])
    #     if m:
    #         fn_name_to_id[m.group(1)] = item["id"]
    #
    # for item in corpus:
    #     for called_fn, called_id in fn_name_to_id.items():
    #         if called_fn + "(" in item["content"] and called_id != item["id"]:
    #             if not G.has_edge(item["id"], called_id):
    #                 G.add_edge(item["id"], called_id, type="calls", weight=0.7)

    # ── Stats ─────────────────────────────────────────────────────────────────
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    edge_types = defaultdict(int)
    for _, _, data in G.edges(data=True):
        edge_types[data.get("type", "unknown")] += 1
    for etype, count in sorted(edge_types.items()):
        print(f"  {etype:12s}: {count} edges")
    return G


G = build_code_graph(corpus)

## 3 · GraphRAG class

**No changes needed here.** This cell is complete and runnable.

How it works:
1. Call `retrieve_complex()` to get initial seed chunks
2. Walk the graph from each seed up to `HOP_DEPTH` hops
3. Rerank seeds + neighbours with the cross-encoder
4. Return top-k

In [ ]:
class GraphRAG:
    HOP_DEPTH      = 1   # hops from each seed — increase for wider expansion
    MAX_NEIGHBOURS = 5   # max neighbours per seed — prevents result explosion

    def __init__(self, rag_instance: VersionControlRAG,
                 corpus_path: str = "local_corpus.json"):
        self.rag = rag_instance
        with open(corpus_path) as f:
            corp = json.load(f)
        self.id_to_content = {item["id"]: item["content"] for item in corp}
        self.graph = G   # use the graph built in the cell above
        print("GraphRAG ready.")

    def _expand(self, seed_ids: list[str]) -> dict[str, str]:
        neighbours = {}
        for seed_id in seed_ids:
            if seed_id not in self.graph:
                continue
            visited  = {seed_id}
            frontier = {seed_id}
            for _ in range(self.HOP_DEPTH):
                next_frontier = set()
                for node in frontier:
                    nbrs = sorted(
                        self.graph.neighbors(node),
                        key=lambda n: self.graph[node][n].get("weight", 0),
                        reverse=True,
                    )
                    for nbr in nbrs[:self.MAX_NEIGHBOURS]:
                        if nbr not in visited:
                            visited.add(nbr)
                            next_frontier.add(nbr)
                            content = self.id_to_content.get(nbr, "")
                            if content:
                                neighbours[nbr] = content
                frontier = next_frontier
        return neighbours

    def retrieve_graph(self, query: str, top_k: int = 5,
                       mode: str = "advanced") -> list[str]:
        """Drop-in replacement for retrieve_complex()."""
        seed_contents = self.rag.retrieve_complex(
            query, top_k=max(top_k, 5), mode=mode
        )
        content_to_id = {v: k for k, v in self.id_to_content.items()}
        seed_ids      = [content_to_id[c] for c in seed_contents
                         if c in content_to_id]

        neighbour_map  = self._expand(seed_ids)
        all_candidates = {}
        for c in seed_contents:
            all_candidates[content_to_id.get(c, c)] = c
        all_candidates.update(neighbour_map)

        if not all_candidates:
            return seed_contents[:top_k]

        pairs  = [[query, code] for code in all_candidates.values()]
        scores = self.rag.reranker.predict(pairs)
        ranked = sorted(zip(all_candidates.values(), scores),
                        key=lambda x: x[1], reverse=True)
        return [code for code, _ in ranked[:top_k]]


print("✅ GraphRAG class defined")

## 4 · Load RAG and run a quick test

In [ ]:
PINECONE_KEY = "YOUR_PINECONE_API_KEY"   # ← paste your key

rag = VersionControlRAG(
    pinecone_key = PINECONE_KEY,
    model_path   = "Qwen/Qwen2.5-Coder-7B",
    adapter_path = "Ivan17Ji/qwen-lora-250",
    ingest_only  = False,
)
rag.load_local_corpus(str(DATA_DIR / "local_corpus.json"))

In [ ]:
graph_rag = GraphRAG(rag_instance=rag, corpus_path=str(DATA_DIR / "local_corpus.json"))

query = "How do I share dependencies across FastAPI routes?"
print(f"Query: {query}\n")

baseline = rag.retrieve_complex(query, top_k=3, mode="advanced")
graph    = graph_rag.retrieve_graph(query, top_k=3, mode="advanced")

print("Baseline:")
for i, r in enumerate(baseline):
    fn = re.search(r"(?:async )?def (\w+)", r)
    print(f"  [{i}] {fn.group(1) if fn else '?'}")

print("\nGraph-expanded:")
for i, r in enumerate(graph):
    fn = re.search(r"(?:async )?def (\w+)", r)
    print(f"  [{i}] {fn.group(1) if fn else '?'}")

## 5 · Add to chunking_test.ipynb

Once the quick test above looks good, add Graph RAG to the full
evaluation in `chunking_test.ipynb`.

In the **Section 2** cell of `chunking_test.ipynb`, add at the bottom:

```python
# ── Graph RAG retriever ────────────────────────────────────────
# Run graph_rag.ipynb first to build the graph and GraphRAG class,
# then import it here.
import importlib, graph_rag as gr_module
importlib.reload(gr_module)
from graph_rag import GraphRAG, build_code_graph, G

graph_rag_instance = GraphRAG(rag_instance=rag)
```

Then in the **Section 3** eval loop, add a new retriever entry:

```python
# The GraphRAG retriever uses Pinecone — plug it in alongside the
# DenseRetriever objects by adding a wrapper
class GraphRetrieverWrapper:
    def retrieve(self, query, top_k):
        return graph_rag_instance.retrieve_graph(query, top_k=top_k, mode="advanced")

retrievers["6_graph_rag"] = GraphRetrieverWrapper()
```